In [1]:
from pathlib import Path
import pickle
import torch
import os
from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_data.datamodule import CrystDataModule
from chggen.pl_modules.model import CHGGen
from chggen.common.data_utils import get_scaler_from_data_list
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.sampler import SubsetRandomSampler
from torch_geometric.data import Batch
import numpy as np
import pytorch_lightning as pl

from types import SimpleNamespace

from pymatgen.core import Structure, Lattice, Species, Element
from pymatgen.core.periodic_table import Element



def get_scaler(dataset, use_prop_scaler = False, 
               scaler_path = None):
    # Load once to compute property scaler
    if scaler_path is None:
        lattice_scaler = get_scaler_from_data_list(
            dataset.cached_data,
            key='scaled_lattice')
        if use_prop_scaler:
            NotImplementedError("Not implemented the multi prop scaler yet.")
    else:
        lattice_scaler = torch.load(
            Path(scaler_path) / 'lattice_scaler.pt')
    return lattice_scaler



/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = CHGNetDataset(path= '/home/zhongpc/chggen/data/mptrj/MPtrj_debug.csv',
                        name = 'mptrj_debug',
                        prop_list = ['e_hull'],
                        )

lattice_scaler = get_scaler(dataset= dataset)


  0%|                                                                                                                        | 0/2417 [00:00<?, ?it/s]/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite p

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
  5%|█████▎                                                                                                       | 117/2417 [00:00<00:07, 306.21it/s]/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite p

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-p

 20%|█████████████████████▋                                                                                       | 480/2417 [00:01<00:05, 360.27it/s]/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
 21%|███████████████████████▎                                                                                     | 517/2417 [00:01<00:05, 338.20it/s]/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatge

 24%|██████████████████████████▍                                                                                  | 585/2417 [00:01<00:05, 322.83it/s]/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
 26%|████████████████████████████▏                                                                                | 626/2417 [00:01<00:05, 345.29it/s]/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatge

...

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-p

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-p

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-p

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/io/cif.py:1120: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-p

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2417/2417 [00:06<00:00, 372.99it/s]
/home/zhongpc/chggen/chggen/common/data_utils.py:647: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  targets = torch.tensor([d[key] for d in data_list])
/home/zhongpc/chggen/chggen/common/data_utils.py:615: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


In [3]:
checkpoint = torch.load('./test_models/chggen.ckpt')


In [4]:
model_hparams ={'latent_dim': 64, 'hidden_dim': 128, 
                'predict_property': True, 'property_dim': 1, # predict the multiple property 
                'load_pretrain': True, 'fc_num_layers': 1, 
                'sigma_begin': 10.0, 'sigma_end': 0.01, 'type_sigma_begin': 5.0, 'type_sigma_end': 0.01,
                'max_atoms': 20, # should be larger than the training set.
                'num_noise_level': 50, 
                'lattice_scale_method': 'scale_length', 
                'cost_natom': 1.0, 'cost_coord': 10.0, 'cost_type': 1.0, 'cost_lattice': 10.0, 'cost_composition': 1.0, 'cost_edge': 10.0, 'cost_property': 1.0,
                'beta': 0.01,
                'teacher_forcing_lattice': True,
                'teacher_forcing_max_epoch': 1000,
                'decoder': 'nequip'}

chggen = CHGGen(lattice_scaler= lattice_scaler, 
                hparams_dict= model_hparams)

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


In [5]:
ld_kwargs = SimpleNamespace(n_step_each = 10,
                            step_lr = 1e-4,
                            min_sigma = 0,
                            save_traj = False,
                            disable_bar = False)


# new_model.langevin_dynamics()

In [6]:
z = torch.rand(3, 64)

results = chggen.langevin_dynamics(z = z, ld_kwargs= ld_kwargs)

/home/zhongpc/chggen/chggen/common/data_utils.py:625: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:27<00:00,  1.81it/s]


In [7]:
lengths = results['lengths']
angles= results['angles']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

In [8]:
batch = torch.arange(len(num_atoms))
batch = batch.repeat_interleave(num_atoms)

In [9]:
batch.size()

torch.Size([28])

In [10]:
num_atoms

tensor([ 9, 14,  5])

In [11]:
for ii in range(len(num_atoms)):
    indices = torch.where(batch == ii)[0]
    print(ii, indices)
    if len(indices) == 0:
        continue
    
    
    Latt = Lattice.from_parameters(a = lengths[ii,0], b = lengths[ii,1], c = lengths[ii,2],
                                   alpha= angles[ii, 0], beta= angles[ii,1], gamma=angles[ii, 2])
                                   
    frac_ = frac_coords[indices]
    type_ = atom_types[indices]
    species_ = [Element.from_Z(ele_Z) for ele_Z in type_]
    
    s_gen = Structure(lattice= Latt , species= species_, coords= frac_,
                      to_unit_cell=False,coords_are_cartesian=False);
    print(s_gen.composition)
    s_gen.to(filename= './test_models/structures/test_' + str(ii) + '.cif')
    

0 tensor([0, 1, 2, 3, 4, 5, 6, 7, 8])
Be1 Pd1 Sn1 Co1 Fe1 Er1 Ar1 Am1 Ac1
1 tensor([ 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22])
N2 Sn1 Ne1 Tb1 Co2 Pd2 Fr2 Er1 Nd1 Np1
2 tensor([23, 24, 25, 26, 27])
P1 Be2 Nd1 F1


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/core/periodic_table.py:221: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  warnings.warn(
/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/pymatgen/core/periodic_table.py:221: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  warnings.warn(


In [12]:
z

tensor([[0.5100, 0.8550, 0.6460, 0.0795, 0.6278, 0.7876, 0.1830, 0.9568, 0.3361,
         0.0205, 0.3039, 0.0318, 0.3866, 0.5289, 0.4804, 0.1556, 0.7942, 0.5302,
         0.7101, 0.6163, 0.4317, 0.8496, 0.5490, 0.1457, 0.7824, 0.0198, 0.2610,
         0.6235, 0.7423, 0.9673, 0.7606, 0.3646, 0.8473, 0.4684, 0.7783, 0.0353,
         0.2958, 0.3610, 0.2100, 0.7785, 0.0334, 0.2756, 0.3624, 0.1819, 0.9879,
         0.0091, 0.3292, 0.9583, 0.2278, 0.5817, 0.8380, 0.3216, 0.9453, 0.6875,
         0.3458, 0.8076, 0.8941, 0.7435, 0.9400, 0.7238, 0.4753, 0.2883, 0.5281,
         0.7243],
        [0.9456, 0.2160, 0.6685, 0.5642, 0.8914, 0.1901, 0.9846, 0.9607, 0.7353,
         0.4239, 0.6285, 0.2986, 0.0954, 0.9138, 0.7055, 0.7881, 0.8860, 0.8751,
         0.8226, 0.3262, 0.6269, 0.6312, 0.1155, 0.9273, 0.4438, 0.4958, 0.4449,
         0.7407, 0.4074, 0.2172, 0.9589, 0.9835, 0.4040, 0.0159, 0.3067, 0.6084,
         0.0762, 0.6785, 0.0926, 0.8105, 0.9018, 0.4411, 0.2351, 0.4006, 0.9442,
         0

In [22]:
c = chggen.predict_property(z)
c_batch = c.repeat_interleave(num_atoms)

In [24]:
c_batch.shape

torch.Size([28])

In [26]:
frac_coords.shape

torch.Size([28, 3])

In [31]:
q_grad = torch.autograd.grad(c_batch, frac_coords, grad_outputs = torch.ones_like(c_batch),
                                        create_graph=True, retain_graph=True)

RuntimeError: One of the differentiated Tensors does not require grad

In [33]:
frac_coords.requires_grad

False

In [34]:
c_batch.requires_grad

True